### ensemble network

#### 모델 내부
0. 핵심은 action head 외에 weight head 추가
1. ACT 구조는 그대로. decoder는 action 뽑는 자리에 추가로 logits 숫자를 뽑음. (아무 숫자를 그냥 출력함)
2. (actions, logits) 쌍이 chunk마다 생성됨

#### 학습
1. 연속된 여러 step에 대해 chunk들 예측
2. 똑같이 softmax ensemble로 ^a계산
3. loss = L1(^a, ground_truth)
4. 이 loss에서 시작해서
   loss -> ^a -> softmax(logits) -> weight head 파라미터
   loss -> ^a -> actions -> action head 파라미터
   -> 양쪽 모두에 gradient 흐름 -> action head와 weight head 동시 학습
   
#### 추론
1. n_action_steps=1로 매 step마다 새 chunk를 예측
2. 버퍼에 (actions, logits) 누적
3. 각 시점에서 겹치는 logits를 softmax -> 가중합 action 출력

### 0. 경로 설정

In [1]:
BASE_DIR = "lerobot_project"
VENV_NAME = "lerobot_env"                                                                                                                               
PYTHON_VERSION = "3.12"                                                                                                                                 
REPO_NAME = "lerobot-ac-mlp"
                                                                                                                                                      
import os, sys, subprocess                                                                                                                              
os.environ['MPLBACKEND'] = 'Agg'                                                                                                                        
home_dir = os.path.expanduser('~')                                                                                                                      
base_path = os.path.join(home_dir, BASE_DIR)                                                                                                            
venv_path = os.path.join(base_path, VENV_NAME)
site_packages = os.path.join(venv_path, 'lib', f'python{PYTHON_VERSION}', 'site-packages')                                                              
if site_packages not in sys.path: sys.path.insert(0, site_packages)                                                                                     
python_exec = os.path.join(venv_path, 'bin', 'python')                                                                                                  
pip_exec = os.path.join(venv_path, 'bin', 'pip')                                                                                                        
project_dir = os.path.join(base_path, REPO_NAME)
src_path = os.path.join(project_dir, 'src')                                                                                                             
if src_path not in sys.path: sys.path.insert(0, src_path)
                                                                                                                                                      
# --- 렌더링 환경 설정 (osmesa) ---                                                                                                                     
conda_lib = os.path.join(home_dir, 'miniconda3', 'lib')                                                                                                 
lib_dirs = {conda_lib}                                                                                                                                  
r = subprocess.run(['bash', '-c', f'find {home_dir}/miniconda3 -name "libOSMesa*" -o -name "libEGL*" 2>/dev/null'],                                     
                  capture_output=True, text=True, timeout=15)                                                                                         
for line in r.stdout.strip().split('\n'):                                                                                                               
  if line.strip(): lib_dirs.add(os.path.dirname(line.strip()))                                                                                        
ld_path = ':'.join(lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = ld_path                                                                                                                 
os.environ['MUJOCO_GL'] = 'osmesa'
                                                                                                                                                      
%cd {project_dir}
print(f"Project: {project_dir}")                                                                                                                        
print(f"Rendering: {os.environ['MUJOCO_GL']}")                                                                                                          
!git branch --show-current 

/home1/eunji24/lerobot_project/lerobot-ac-mlp
Project: /home1/eunji24/lerobot_project/lerobot-ac-mlp
Rendering: osmesa
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
ecd-eunji


### 세팅

1. 기본: save_freq=20000, eval_freq=20000, batch_size=8, step=100k
2. 30k: eval_freq=10000, save_freq=10000, batch_size=2, step=30k
3. new: training step 50k

In [4]:
ENV = "aloha"
TASK = "AlohaTransferCube-v0"                                                                                                                           
DATASET = "lerobot/aloha_sim_transfer_cube_human"                                                                                                       
                                                                                                                                                      
STEPS = 100000          # full: 100000                                                                                                                  
SAVE_FREQ = 20000       # full: 20000                                                                                                                   
BATCH_SIZE = 8                                                                                                                                         
              
# 학습 중 eval 설정 (osmesa 렌더링 사용)                                                                                                                
EVAL_FREQ = 20000       # full: 10000
EVAL_EPISODES = 50                                                                                                                                      
EVAL_BATCH = 1                                                                                                                                          

# 학습된 temporal ensemble 설정                                                                                                                         
USE_LEARNED_ENSEMBLE = True
TRAIN_ENSEMBLE_STEPS = 5   # L: 학습 시 연속 obs 개수                                                                                                   
N_ACTION_STEPS = 1         # learned ensemble 시 무조건 1                                                                                               
                                                                                                                                                      
print("실험 설정")                                                                                                                                      
print("=" * 60)                                                                                                                                         
print(f"  환경:       {ENV} / {TASK}")                                                                                                                  
print(f"  데이터셋:   {DATASET}")                                                                                                                       
print(f"  스텝:       {STEPS},  저장: 매 {SAVE_FREQ}")
print(f"  배치:       {BATCH_SIZE}")                                                                                                                    
print(f"  평가:       매 {EVAL_FREQ} 스텝, {EVAL_EPISODES} episodes")
print(f"  렌더링:     {os.environ['MUJOCO_GL']}")                                                                                                       
print(f"  Learned TE: {USE_LEARNED_ENSEMBLE}, L={TRAIN_ENSEMBLE_STEPS}")                                                                                
print("=" * 60)                                                                          

실험 설정
  환경:       aloha / AlohaTransferCube-v0
  데이터셋:   lerobot/aloha_sim_transfer_cube_human
  스텝:       100000,  저장: 매 20000
  배치:       8
  평가:       매 20000 스텝, 50 episodes
  렌더링:     osmesa
  Learned TE: True, L=5


### training

1. act_learned_ensemble_L2_100k : 처음 decoder out 분리 전
2. act_learned_ensemble_L2 : decoder out 분리
3. act_learned_ensemble_L32_30k : decoder out 분리, L=32, step=30k
4. act_learned_ensemble_L32_30k_baseline: act_learned_ensemble_L32_30k baseline
5. act_learned_ensemble_L10_50k: 

ACT (learned TE) 학습 시작...
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
INFO 2026-04-29 01:05:31 ot_train.py:164 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
          

평가

In [3]:
import os, subprocess, sys              
                                                                                       
LOGIT_LOG = "/tmp/logit_diag_act_learned_40k"   # 진단 결과 저장 경로                    
                                      
cmd = [                                                                                  
  python_exec,                                                                         
  "-m", "lerobot.scripts.lerobot_eval",                                                
  # 본인의 learned ensemble 체크포인트 경로로 수정                                     
  "--policy.path=outputs/train/act_learned_ensemble_L2_100k/checkpoints/040000/pretrained_model",                                                                               
  "--policy.n_action_steps=1",                                                         
  "--policy.use_learned_ensemble=true",                                                
  "--env.type=aloha",                                                                  
  "--env.task=AlohaTransferCube-v0",                                                   
  "--eval.n_episodes=5",                       # 진단용이라 5개만                      
  "--eval.batch_size=1",                                                               
  "--output_dir=outputs/eval/act_learned_diag",
]                                                                                        
                                                                                       
env = os.environ.copy()                                                                  
env["MUJOCO_GL"] = os.environ["MUJOCO_GL"]                                               
env["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]
env["LOGIT_LOG_DIR"] = LOGIT_LOG                  # 진단 로깅 켜기                       
                                          
print("CMD:", " ".join(cmd))                                                             

proc = subprocess.Popen(                                                                 
  cmd,        
  env=env,                                                                             
  stdout=subprocess.PIPE,             
  stderr=subprocess.STDOUT,
  text=True,                                                                           
  bufsize=1,
)                                                                                        
                                      
for line in proc.stdout:
  print(line, end="")                                                                  

proc.wait()                                                                              
print("\nRETURN CODE:", proc.returncode)
print(f"\nLogits saved to: {LOGIT_LOG}")    

CMD: /home1/eunji24/lerobot_project/lerobot_env/bin/python -m lerobot.scripts.lerobot_eval --policy.path=outputs/train/act_learned_ensemble_L2_100k/checkpoints/040000/pretrained_model --policy.n_action_steps=1 --policy.use_learned_ensemble=true --env.type=aloha --env.task=AlohaTransferCube-v0 --eval.n_episodes=5 --eval.batch_size=1 --output_dir=outputs/eval/act_learned_diag


KeyboardInterrupt: 

분석

In [7]:
import numpy as np                                                                       
import glob, os                                                                          
import matplotlib.pyplot as plt             
                                      
LOGIT_LOG = "/tmp/logit_diag_act_learned_40k"   # 위와 동일 경로
                                                                                       
# 모든 logit 파일 로드
files = sorted(glob.glob(f"{LOGIT_LOG}/logits_*.npy"))                                   
print(f"Loaded {len(files)} steps of logits")
                                                                                       
all_logits = np.concatenate([np.load(f) for f in files], axis=0)
print(f"Shape: {all_logits.shape}  (steps, chunk_size)")                                 
                                                                                       
# ===== 통계 =====                                                                       
print("\n" + "=" * 60)                                                                   
print("LOGIT STATS")                                                                     
print("=" * 60)
print(f"Overall mean:          {all_logits.mean():>10.4f}")                              
print(f"Overall std:           {all_logits.std():>10.4f}")
print(f"Min / Max:             {all_logits.min():>10.4f} / {all_logits.max():>10.4f}")
                                          
intra_std = all_logits.std(axis=1).mean()                                                
across_std = all_logits.std(axis=0).mean()
print(f"\nIntra-chunk std (한 chunk 내 위치별 차이):  {intra_std:.4f}")                  
print(f"Across-step std (시점별 변동):              {across_std:.4f}")
                                                                                       
# Softmax 분포                                                                           
probs = np.exp(all_logits - all_logits.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)                                         
entropy = -np.sum(probs * np.log(probs + 1e-9), axis=1)
uniform_entropy = np.log(all_logits.shape[1])                                            
print(f"\nSoftmax entropy 평균:    {entropy.mean():.4f}")
print(f"균등 분포 entropy:        {uniform_entropy:.4f}  ← chunk 내 균등이면 이 값")     
print(f"비율 (균등 대비):         {entropy.mean() / uniform_entropy:.2%}")
                                                                                       
# ===== 진단 =====                          
print("\n" + "=" * 60)                                                                   
print("DIAGNOSIS")
print("=" * 60)                                                                          
ent_ratio = entropy.mean() / uniform_entropy
                                                                                       
if intra_std < 0.01 and across_std < 0.01:
  print("🔴 DEGENERATE: logit이 거의 상수 → weight head 죽음")
  print("   → 균등 평균 = 고정 TE와 동일 동작 → 36% ≈ 34% 일치 설명됨")                
  print("   → decoder 분리 또는 L 키우기 필요")
elif intra_std < 0.05:                                                                   
  print("🟡 PARTIAL DEGENERATE: chunk 내 위치들 logit 거의 같음")                      
  print("   → 위치별 가중치 차별화 안 됨")
elif across_std < 0.05:                                                                  
  print("🟡 OBS-INSENSITIVE: obs 변해도 logit 변화 적음")
  print("   → 상황별 적응 못 함 (mode 충돌 감지 X)")                                   
elif ent_ratio > 0.99:                  
  print("🟡 NEAR-UNIFORM softmax: 거의 균등 → 평균과 거의 같음")                       
elif ent_ratio < 0.10:                                                                   
  print("🟡 OVER-COLLAPSED softmax: 너무 sharp → 한쪽만 사용")
else:                                                                                    
  print("🟢 logit이 다양하게 변동 — weight head 학습됨")
  print("   → 다른 원인 (학습 신호 자체 부정확) 의심")                                 
                                                                                       
# ===== 시각화 =====
fig, axes = plt.subplots(1, 3, figsize=(15, 4))                                          
                                          
# 1. 한 step에서 chunk 위치별 logit                                                      
sample_step = all_logits.shape[0] // 2
axes[0].plot(all_logits[sample_step])                                                    
axes[0].set_title(f"Logits at step {sample_step} (across chunk positions)")
axes[0].set_xlabel("chunk position")                                                     
axes[0].set_ylabel("logit")                 
                                                                                       
# 2. 한 위치(예: 50)의 시간에 따른 변화
axes[1].plot(all_logits[:, 50])                                                          
axes[1].set_title("Logits at chunk position 50 (over time)")
axes[1].set_xlabel("step")                                                               
axes[1].set_ylabel("logit")
                                                                                       
# 3. Softmax entropy 시간 추이              
axes[2].plot(entropy)                                                                    
axes[2].axhline(uniform_entropy, color='r', linestyle='--', label='uniform')
axes[2].set_title("Softmax entropy over time")                                           
axes[2].set_xlabel("step")                  
axes[2].set_ylabel("entropy")                                                            
axes[2].legend()
                                                                                       
plt.tight_layout()                          
plt.savefig(f"{LOGIT_LOG}/diagnosis.png", dpi=100)                                       
plt.show()      
print(f"\nPlot saved: {LOGIT_LOG}/diagnosis.png") 

Loaded 1922 steps of logits
Shape: (1922, 100)  (steps, chunk_size)

LOGIT STATS
Overall mean:              1.3704
Overall std:               0.1601
Min / Max:                 0.7857 /     1.7334

Intra-chunk std (한 chunk 내 위치별 차이):  0.0729
Across-step std (시점별 변동):              0.1562

Softmax entropy 평균:    4.6018
균등 분포 entropy:        4.6052  ← chunk 내 균등이면 이 값
비율 (균등 대비):         99.93%

DIAGNOSIS
🟡 NEAR-UNIFORM softmax: 거의 균등 → 평균과 거의 같음

Plot saved: /tmp/logit_diag_act_learned_40k/diagnosis.png


앙상블 효과 진단

In [8]:
import subprocess, tempfile, os            
                                                                                                                                                                                                  
# 테스트 코드 (venv python에서 실행될)      
test_code = '''                                                                                                                                                                                     
import sys                             
sys.path.insert(0, "/home1/eunji24/lerobot_project/lerobot-ac-mlp/src")                                                                                                                             
import torch                            
from lerobot.policies.act.modeling_act import ACTLearnedTemporalEnsembler                                                                                                                           
                                                                                                                                                                                                  
print("torch version:", torch.__version__)  
print("=" * 60)                                                                                                                                                                                     
print("실제 ACTLearnedTemporalEnsembler 클래스 직접 테스트")
print("=" * 60)                                                                                                                                                                                     
                                          
B = 1                                                                                                                                                                                               
chunk_size = 100                            
action_dim = 14                                                                                                                                                                                     

actions_A = torch.zeros(B, chunk_size, action_dim)                                                                                                                                                  
actions_B = torch.ones(B, chunk_size, action_dim)
                                                                                                                                                                                                  
# Test 1                                   
print()
print("--- Test 1: A에 큰 logit ---")       
ens = ACTLearnedTemporalEnsembler(chunk_size)
out1 = ens.update(actions_A, torch.full((B, chunk_size), 10.0))
print("Step 1:", round(out1[0,0].item(), 4), "(예상 0.0)")                                                                                                                                          
out2 = ens.update(actions_B, torch.full((B, chunk_size), 0.0))
print("Step 2:", round(out2[0,0].item(), 4), "(예상 ~0.0)")                                                                                                                                         
test1 = abs(out2[0,0].item()) < 0.01    
                                          
# Test 2                                                                                                                                                                                            
print()                                     
print("--- Test 2: B에 큰 logit ---")                                                                                                                                                               
ens = ACTLearnedTemporalEnsembler(chunk_size)
ens.update(actions_A, torch.full((B, chunk_size), 0.0))                                                                                                                                             
out2 = ens.update(actions_B, torch.full((B, chunk_size), 10.0))
print("Step 2:", round(out2[0,0].item(), 4), "(예상 ~1.0)")                                                                                                                                         
test2 = abs(out2[0,0].item() - 1.0) < 0.01                                                                                                                                                          
                                          
# Test 3                                                                                                                                                                                            
print()                                     
print("--- Test 3: 균등 logit ---")                                                                                                                                                                 
ens = ACTLearnedTemporalEnsembler(chunk_size)
ens.update(actions_A, torch.zeros(B, chunk_size))                                                                                                                                                   
out2 = ens.update(actions_B, torch.zeros(B, chunk_size))
print("Step 2:", round(out2[0,0].item(), 4), "(예상 0.5)")                                                                                                                                          
test3 = abs(out2[0,0].item() - 0.5) < 0.01
                                          
# Test 4                                                                                                                                                                                            
print()                                     
print("--- Test 4: 매 step 다른 logit ---")                                                                                                                                                         
ens = ACTLearnedTemporalEnsembler(chunk_size)
ens.update(actions_A, torch.full((B, chunk_size), 5.0))                                                                                                                                             
out2 = ens.update(actions_B, torch.full((B, chunk_size), 5.0))
print("Step 2:", round(out2[0,0].item(), 4), "(예상 0.5)")                                                                                                                                          
out3 = ens.update(actions_A, torch.full((B, chunk_size), 100.0))
print("Step 3:", round(out3[0,0].item(), 4), "(예상 ~0.0)")
test4 = abs(out3[0,0].item()) < 0.01                                                                                                                                                                
                                         
print()                                                                                                                                                                                             
print("=" * 60)                                                                                                                                                                                     
print("Test 1:", "PASS" if test1 else "FAIL")
print("Test 2:", "PASS" if test2 else "FAIL")                                                                                                                                                       
print("Test 3:", "PASS" if test3 else "FAIL")                                                                                                                                                       
print("Test 4:", "PASS" if test4 else "FAIL")                                                                                                                                                       
if all([test1, test2, test3, test4]):                                                                                                                                                               
  print()                                
  print("[ALL PASS] 코드가 의도대로 동작")
  print("  -> 큰 logit이 우세, 매 step 새 logit 적용됨")                                                                                                                                          
  print("  -> 코드 버그 아님. 학습 신호 부족이 진짜 원인")
else:                                                                                                                                                                                               
  print()                                                                                                                                                                                         
  print("[FAIL] 코드 버그")               
'''                                                                                                                                                                                                 
                                                                                                                                                                                                  
# 임시 파일에 쓰고 venv python으로 실행                                                                                                                                                             
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:                                                                                                                        
  f.write(test_code)                                                                                                                                                                              
  tmp_path = f.name                                                                                                                                                                               
                                      
result = subprocess.run(                   
  [python_exec, tmp_path],                                                                                                                                                                        
  capture_output=True, text=True          
)                                                                                                                                                                                                   
                                     
print(result.stdout)                                                                                                                                                                                
if result.returncode != 0:                 
  print("STDERR:")                                                                                                                                                                                
  print(result.stderr)                    
                                      
os.unlink(tmp_path)
                           

torch version: 2.7.1+cu128
실제 ACTLearnedTemporalEnsembler 클래스 직접 테스트

--- Test 1: A에 큰 logit ---
Step 1: 0.0 (예상 0.0)
Step 2: 0.0 (예상 ~0.0)

--- Test 2: B에 큰 logit ---
Step 2: 1.0 (예상 ~1.0)

--- Test 3: 균등 logit ---
Step 2: 0.5 (예상 0.5)

--- Test 4: 매 step 다른 logit ---
Step 2: 0.5 (예상 0.5)
Step 3: nan (예상 ~0.0)

Test 1: PASS
Test 2: PASS
Test 3: PASS
Test 4: FAIL

[FAIL] 코드 버그



In [3]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

LOG_DIR = "/tmp/chunks_log_TE001_50ep"
chunk_size = 100

# 1) chunk 파일 로드
files = sorted(glob.glob(os.path.join(LOG_DIR, "chunk_*.npy")))

print(f"LOG_DIR: {LOG_DIR}")
print(f"Loaded {len(files)} chunks (one per timestep)")

if len(files) == 0:
    raise FileNotFoundError(
        f"No chunk_*.npy files found in {LOG_DIR}. "
        "Check LOG_DIR path or whether logging actually saved chunk files."
    )

# 2) 모든 chunk를 하나의 array로 합치기
loaded_chunks = []
bad_files = []

for f in files:
    try:
        arr = np.load(f)
        loaded_chunks.append(arr)
    except Exception as e:
        bad_files.append((f, str(e)))

if bad_files:
    print("\nSome files failed to load:")
    for f, e in bad_files[:5]:
        print(f"  {f}: {e}")
    raise RuntimeError("Some .npy files could not be loaded.")

chunks = np.stack(loaded_chunks)

print("Raw chunks shape:", chunks.shape)

# 예상 shape: (T, B, chunk_size, action_dim)
if chunks.ndim != 4:
    raise ValueError(
        f"Expected chunks shape (T, B, chunk_size, action_dim), "
        f"but got {chunks.shape}"
    )

T, B, C, A = chunks.shape
print(f"Shape: T={T}, B={B}, chunk_size={C}, action_dim={A}")

if C != chunk_size:
    print(f"Warning: given chunk_size={chunk_size}, but loaded C={C}. Using C={C}.")
    chunk_size = C

# 3) B개 env 각각을 episode로 보고 candidate variance 분석
all_episodes_data = []

for b in range(B):
    ep_chunks = chunks[:, b]  # (T, chunk_size, action_dim)

    variances = []
    dist_best = []
    dist_worst = []
    valid_taus = []

    for tau in range(T):
        cands = []

        # tau를 예측할 수 있는 chunk 시작점 k 범위
        start_k = max(0, tau - chunk_size + 1)
        end_k = min(T - 1, tau)

        for k in range(start_k, end_k + 1):
            offset = tau - k
            if 0 <= offset < chunk_size:
                cands.append(ep_chunks[k, offset])

        if len(cands) >= 2:
            cands = np.stack(cands)  # (num_candidates, action_dim)

            mean_action = cands.mean(axis=0)

            # candidate spread
            variances.append(cands.std(axis=0).mean())

            # mean과 각 candidate 사이 거리
            dists = np.abs(cands - mean_action).mean(axis=1)
            dist_best.append(dists.min())
            dist_worst.append(dists.max())
            valid_taus.append(tau)

    if len(variances) > 0:
        all_episodes_data.append({
            "episode": b,
            "tau": np.array(valid_taus),
            "variance": np.array(variances),
            "dist_to_best": np.array(dist_best),
            "dist_to_worst": np.array(dist_worst),
        })

print(f"Episodes analyzed: {len(all_episodes_data)}")

if len(all_episodes_data) == 0:
    raise RuntimeError(
        "No valid episode data. This usually means there were fewer than 2 candidates "
        "for every timestep. Check T, B, chunk_size, and saved chunk format."
    )

# 4) 전체 통계
all_vars = np.concatenate([r["variance"] for r in all_episodes_data])
all_dist_best = np.concatenate([r["dist_to_best"] for r in all_episodes_data])
all_dist_worst = np.concatenate([r["dist_to_worst"] for r in all_episodes_data])

print("\n" + "=" * 60)
print("MODE COLLAPSE 분석")
print("=" * 60)
print(f"Total valid timesteps: {len(all_vars)}")

print("\nCandidate variance:")
print(f"  평균:        {all_vars.mean():.6f}")
print(f"  최대:        {all_vars.max():.6f}")
print(f"  90th %ile:   {np.percentile(all_vars, 90):.6f}")
print(f"  99th %ile:   {np.percentile(all_vars, 99):.6f}")

print("\nMean-to-candidate:")
print(f"  nearest 평균:  {all_dist_best.mean():.6f}")
print(f"  farthest 평균: {all_dist_worst.mean():.6f}")

# 5) 진단
print("\n" + "=" * 60)
print("DIAGNOSIS")
print("=" * 60)

p99 = np.percentile(all_vars, 99)
max_var = all_vars.max()

if p99 > 0.1:
    print("🔴 일부 timestep에서 candidate spread가 큼 → action 후보들이 많이 갈라지는 정황")
elif max_var > 0.05:
    print("🟡 일부 timestep에서만 candidate spread가 약간 큼")
else:
    print("🟢 candidate들이 대체로 비슷함 → 큰 mode collapse 정황은 약함")

if all_dist_best.mean() > all_vars.mean() * 0.7:
    print("⚠️ 평균 action이 candidate 중심에서 다소 떨어질 수 있음")
else:
    print("✅ 평균 action이 candidate 중심에 비교적 가까움")

# 6) 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

ep_idx = 0
ep_data = all_episodes_data[ep_idx]

axes[0, 0].plot(ep_data["tau"], ep_data["variance"])
axes[0, 0].set_title(f"Episode {ep_data['episode']}: variance per tau")
axes[0, 0].set_xlabel("tau")
axes[0, 0].set_ylabel("std")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(all_vars, bins=80, log=True)
axes[0, 1].set_title("Variance distribution")
axes[0, 1].set_xlabel("std")
axes[0, 1].set_ylabel("log count")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(ep_data["tau"], ep_data["dist_to_best"], label="nearest", alpha=0.7)
axes[1, 0].plot(ep_data["tau"], ep_data["dist_to_worst"], label="farthest", alpha=0.7)
axes[1, 0].set_title(f"Episode {ep_data['episode']}: mean-to-candidate")
axes[1, 0].set_xlabel("tau")
axes[1, 0].set_ylabel("distance")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

max_vars = [r["variance"].max() for r in all_episodes_data]
axes[1, 1].hist(max_vars, bins=20)
axes[1, 1].set_title("Max variance per episode")
axes[1, 1].set_xlabel("max std")
axes[1, 1].set_ylabel("count")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()

save_path = os.path.join(LOG_DIR, "mode_collapse.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
print(f"\nSaved figure to: {save_path}")

plt.show()

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [7]:
import numpy as np                                                                          
import glob                                                                                 
import matplotlib.pyplot as plt                                                             
                                                                                          
LOG_DIR = "/tmp/chunks_log_TE001_50ep"     # ← 위와 동일하게                                
chunk_size = 100                                                                            

files = sorted(glob.glob(f"{LOG_DIR}/episode_*.npz"))                                       
print(f"Loaded {len(files)} batched files")

all_episodes_data = []                                                                      
                                      
for f in files:                                                                             
  data = np.load(f)                                                                       
  chunks = data['chunks']    # (T, B, chunk_size, action_dim)
  T, B = chunks.shape[:2]                                                                 
                                                                                          
  for b in range(B):                
      ep_chunks = chunks[:, b]    # (T, chunk_size, action_dim)                           
                                          
      variances, dist_best, dist_worst = [], [], []
      for tau in range(T):                                                                
          cands = []                                                                      
          for k in range(max(0, tau - chunk_size + 1), min(T - 1, tau) + 1):              
              cands.append(ep_chunks[k, tau - k])                                         
          if len(cands) >= 2:                                                             
              cands = np.stack(cands)                                                     
              m = cands.mean(axis=0)                                                      
              variances.append(cands.std(axis=0).mean())                                  
              dists = [np.abs(c - m).mean() for c in cands]                               
              dist_best.append(min(dists))                                                
              dist_worst.append(max(dists))                                               
                                                                                          
      if len(variances) > 0:          
          all_episodes_data.append({                                                      
              'variance': np.array(variances),                                            
              'dist_to_best': np.array(dist_best),
              'dist_to_worst': np.array(dist_worst),                                      
          })                         
                                                                                          
print(f"Total episodes analyzed: {len(all_episodes_data)}")
                                    
all_vars = np.concatenate([r['variance'] for r in all_episodes_data])                       
all_dist_best = np.concatenate([r['dist_to_best'] for r in all_episodes_data])
all_dist_worst = np.concatenate([r['dist_to_worst'] for r in all_episodes_data])            
                                          
print("\n" + "=" * 60)                                                                      
print("MODE COLLAPSE 분석")           
print("=" * 60)                                                                             
print(f"Total timesteps: {len(all_vars)}")  
print(f"\nCandidate variance:")                                                             
print(f"  평균:        {all_vars.mean():.4f}")                                              
print(f"  최대:        {all_vars.max():.4f}")                                               
print(f"  90th %ile:   {np.percentile(all_vars, 90):.4f}")                                  
print(f"  99th %ile:   {np.percentile(all_vars, 99):.4f}")                                  
print(f"\nMean-to-candidate:")              
print(f"  → best 평균:  {all_dist_best.mean():.4f}")                                        
print(f"  → worst 평균: {all_dist_worst.mean():.4f}")
                                                                                          
# === 진단 ===                         
print("\n" + "=" * 60)                                                                      
print("DIAGNOSIS")                          
print("=" * 60)                                                                             
if np.percentile(all_vars, 99) > 0.1:                                                       
  print(f"🔴 일부 τ에서 큰 candidate spread → mode collapse 정황 있음")                   
elif all_vars.max() > 0.05:                                                                 
  print(f"🟡 일부 시점에서만 약간 갈림")                                                  
else:                                  
  print(f"🟢 candidate 거의 균일 → mode collapse 거의 발생 X")                            
                                          
if all_dist_best.mean() > all_vars.mean() * 0.7:                                            
  print(f"⚠️  평균이 candidates 중심에서 떨어진 경향")                                     
else:                                                                                       
  print(f"✅ 평균이 candidates 중심에 가까움")                                            
                                                                                          
# === 시각화 ===                                                                            
fig, axes = plt.subplots(2, 2, figsize=(14, 8))                                             
ep_idx = 0                                                                                  
                                                                                          
axes[0, 0].plot(all_episodes_data[ep_idx]['variance'])
axes[0, 0].set_title(f'Episode 0: variance per τ')                                          
axes[0, 0].set_xlabel('τ')              
axes[0, 0].set_ylabel('std')                                                                
axes[0, 0].grid(True, alpha=0.3)                                                            
                                          
axes[0, 1].hist(all_vars, bins=80, log=True)                                                
axes[0, 1].set_title(f'Variance dist (all timesteps)')
axes[0, 1].set_xlabel('std')                                                                
axes[0, 1].set_ylabel('log count')      
axes[0, 1].grid(True, alpha=0.3)                                                            
                                                                                          
axes[1, 0].plot(all_episodes_data[ep_idx]['dist_to_best'], label='nearest', alpha=0.7)
axes[1, 0].plot(all_episodes_data[ep_idx]['dist_to_worst'], label='farthest', alpha=0.7)    
axes[1, 0].fill_between(                                                                    
  range(len(all_episodes_data[ep_idx]['dist_to_best'])),
  all_episodes_data[ep_idx]['dist_to_best'],                                              
  all_episodes_data[ep_idx]['dist_to_worst'], alpha=0.2,
)                                                                                           
axes[1, 0].set_title(f'Episode 0: mean-to-candidate distance')
axes[1, 0].set_xlabel('τ'); axes[1, 0].set_ylabel('distance')                               
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)                                       
                                                                                          
max_vars = [r['variance'].max() for r in all_episodes_data]                                 
axes[1, 1].hist(max_vars, bins=20)                                                          
axes[1, 1].set_title('Max variance per episode')
axes[1, 1].set_xlabel('max std'); axes[1, 1].set_ylabel('count')                            
axes[1, 1].grid(True, alpha=0.3)                                                            
                                                                                          
plt.tight_layout()                                                                          
plt.savefig(f"{LOG_DIR}/mode_collapse_analysis.png", dpi=120, bbox_inches='tight')
plt.show()                                                                                  
print(f"\nPlot: {LOG_DIR}/mode_collapse_analysis.png")
                     

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [9]:
import sys
import subprocess

python = sys.executable

subprocess.check_call([python, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "matplotlib"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 44.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 100.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 117.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 86.5 MB/s  0:00:00
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: packagingm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/11 [pillow]ng]
    Found existing installation: packaging 25.0━━━━━━━━━━━━━━━  2/11 [pillow]
    Uninstalling packaging-25.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/11 [pillow]
      Successfully uninstalled packaging-25.0━━━━━━━━━━━━━━━━━  2/11 [pillow]
  Attempting uninstall: numpy━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/11 [packaging]
    Found existing installation: numpy 2.1.3━━━━━━━━━━━━━━━━━━  3/11 [packa

0